# UPP smoke: A vs F0 vs FX (100 steps, 2x T4)

Decision only: does the block + xattn readout (FX) learn above chance, and does it beat the Kev pointer (A) and the block-dot-without-request (F0)?

Hard cap: 100 steps, 400 train rows, no ToolRet, no Gemma. If this is not clearly above chance, do not start a long run.

In [ ]:
import os, subprocess, sys, time
REPO_URL = "https://github.com/nisaral/Soro-Boro.git"
BRANCH = "upp-p0"
WORK = "/kaggle/working" if os.path.isdir("/kaggle") else "/content"
REPO = f"{WORK}/Soro-Boro"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO], check=True)
os.chdir(REPO)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout, flush=True)

In [ ]:
import torch
!pip -q install -U "transformers>=5.0" peft bitsandbytes accelerate huggingface_hub safetensors
!nvidia-smi --query-gpu=name,memory.total --format=csv
print(torch.__version__, torch.cuda.device_count(), "GPU(s)", flush=True)
assert torch.cuda.device_count() >= 1, "need a GPU"

In [ ]:
COMMON = (
    "--base-model Qwen/Qwen3-0.6B-Base --train-rows 400 --eval-rows 80 --batch 4 "
    "--max-seq-len 832 --max-steps 100 --log-every 20"
)
t0 = time.time()
gpu0 = subprocess.Popen(
    f"CUDA_VISIBLE_DEVICES=0 python veyra/training/p0_upp_ablation.py --configs A,F0 {COMMON} --out runs/p0_smoke0 > runs_p0_smoke0.log 2>&1",
    shell=True,
)
gpu1 = subprocess.Popen(
    f"CUDA_VISIBLE_DEVICES=1 python veyra/training/p0_upp_ablation.py --configs FX {COMMON} --out runs/p0_smoke1 > runs_p0_smoke1.log 2>&1",
    shell=True,
)
print("started A,F0 on GPU0 and FX on GPU1", flush=True)
gpu0.wait(); gpu1.wait()
print(f"done in {(time.time()-t0)/60:.1f} min  rc={gpu0.returncode},{gpu1.returncode}", flush=True)
print(open("runs_p0_smoke0.log").read()[-3000:])
print("---")
print(open("runs_p0_smoke1.log").read()[-3000:])

In [ ]:
import json, pathlib
res = {}
for d in ("runs/p0_smoke0", "runs/p0_smoke1"):
    p = pathlib.Path(d) / "results.json"
    if p.exists():
        res.update(json.loads(p.read_text()))
pathlib.Path("runs").mkdir(exist_ok=True)
pathlib.Path("runs/p0_smoke.json").write_text(json.dumps(res, indent=2))
sys.path.insert(0, "veyra/training")
from p0_upp_ablation import markdown
print(markdown(dict(sorted(res.items()))))
chance = 0.22
for k, v in res.items():
    acc = (v.get("eval") or {}).get("selection", {}).get("acc")
    print(f"GATE {k} selection acc={acc}  above_chance={acc is not None and acc > chance}")